<a href="https://colab.research.google.com/github/Mohamad-101/Secure-Attendance-System-Using-Facial-Recognition/blob/main/research_sandbox/Phase1_Vision_and_DB_Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 Vision and Database Tests

This notebook tests the Phase 1 prototype for the Secure Attendance System with Face Authentication.

It demonstrates:
- loading face images
- generating facial encodings
- comparing face distances
- creating SQLite database tables
- inserting a demo attendance record

The notebook is designed for Google Colab because some face recognition libraries may be difficult to run on low-spec local devices.

In [1]:
!pip install face_recognition opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=398c596da579cb1c2dc48b9bd29623ea8d79756ceded0253c9bd76dc6ac365bd
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models


In [2]:
import face_recognition
import sqlite3
import json
import numpy as np
from datetime import date
from google.colab import files

print("Environment setup completed.")

Environment setup completed.


In [7]:
uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print(filename)

Saving imposter.jpg to imposter.jpg
Saving verification.jpeg to verification.jpeg
Saving baseline.jpeg to baseline.jpeg
Uploaded files:
imposter.jpg
verification.jpeg
baseline.jpeg


In [8]:
def extract_single_face_encoding(image_path):
    image = face_recognition.load_image_file(image_path)
    encodings = face_recognition.face_encodings(image)

    if len(encodings) == 0:
        raise ValueError(f"No face detected in {image_path}")

    if len(encodings) > 1:
        raise ValueError(f"Multiple faces detected in {image_path}. Use one face only.")

    return encodings[0]

In [10]:
baseline_encoding = extract_single_face_encoding("baseline.jpeg")
verification_encoding = extract_single_face_encoding("verification.jpeg")
imposter_encoding = extract_single_face_encoding("imposter.jpg")

print("Face encodings generated successfully.")
print("Encoding length:", len(baseline_encoding))

Face encodings generated successfully.
Encoding length: 128


In [11]:
same_user_distance = face_recognition.face_distance(
    [baseline_encoding],
    verification_encoding
)[0]

imposter_distance = face_recognition.face_distance(
    [baseline_encoding],
    imposter_encoding
)[0]

threshold = 0.60

print("Same user distance:", same_user_distance)
print("Imposter distance:", imposter_distance)

print("Same user accepted:", same_user_distance <= threshold)
print("Imposter accepted:", imposter_distance <= threshold)

Same user distance: 0.5288507307473901
Imposter distance: 0.7870910655102877
Same user accepted: True
Imposter accepted: False


In [12]:
conn = sqlite3.connect("phase1_attendance_demo.db")
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    email TEXT UNIQUE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Facial_Profiles (
    profile_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    face_encoding TEXT NOT NULL,
    enrollment_date DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE CASCADE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Attendance_Sessions (
    session_id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_name TEXT NOT NULL,
    session_date DATE NOT NULL,
    start_time TIME,
    end_time TIME
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Attendance_Logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    session_id INTEGER,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
    attendance_status TEXT DEFAULT 'present',
    face_distance REAL,
    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE CASCADE,
    FOREIGN KEY (session_id) REFERENCES Attendance_Sessions(session_id) ON DELETE SET NULL
);
""")

conn.commit()

print("Database tables created successfully.")

Database tables created successfully.


In [13]:
cursor.execute("""
INSERT INTO Users (full_name, email)
VALUES (?, ?)
""", ("Demo User", "demo.user@example.com"))

user_id = cursor.lastrowid

serialized_encoding = json.dumps(baseline_encoding.tolist())

cursor.execute("""
INSERT INTO Facial_Profiles (user_id, face_encoding)
VALUES (?, ?)
""", (user_id, serialized_encoding))

cursor.execute("""
INSERT INTO Attendance_Sessions (session_name, session_date, start_time, end_time)
VALUES (?, ?, ?, ?)
""", ("Phase 1 Demo Session", str(date.today()), "10:00", "12:00"))

session_id = cursor.lastrowid

if same_user_distance <= threshold:
    cursor.execute("""
    INSERT INTO Attendance_Logs (user_id, session_id, attendance_status, face_distance)
    VALUES (?, ?, ?, ?)
    """, (user_id, session_id, "present", float(same_user_distance)))

conn.commit()

print("Demo user, facial profile, session, and attendance log inserted.")

Demo user, facial profile, session, and attendance log inserted.


In [14]:
cursor.execute("""
SELECT
    Users.full_name,
    Attendance_Sessions.session_name,
    Attendance_Logs.timestamp,
    Attendance_Logs.attendance_status,
    Attendance_Logs.face_distance
FROM Attendance_Logs
JOIN Users ON Attendance_Logs.user_id = Users.user_id
LEFT JOIN Attendance_Sessions
ON Attendance_Logs.session_id = Attendance_Sessions.session_id;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

('Demo User', 'Phase 1 Demo Session', '2026-06-16 15:36:10', 'present', 0.5288507307473901)


## Phase 1 Result

This notebook confirms that the Phase 1 prototype can:

- generate facial encodings
- compare a verified user against an imposter
- create the planned SQLite database tables
- store a demo facial profile
- create a demo attendance session
- insert an attendance log after successful verification

Full registration, dashboard, reporting, and anti-spoofing will be developed in later phases.